# Steam Reviews 2021 — Data Mining Final Project
## Notebook 04 · Part 2, section D: Text Mining and NLP (+ sections 6–8 for the text models)

This is the **main section of the project** (project type chosen in 1.5: *Text Mining*).

| Subsection | Assignment requirement |
|---|---|
| D.1 | Clean and preprocess the text (tokenisation, stop words, vocabulary) |
| D.2 | Use **Bag-of-Words or TF-IDF** — both are built and compared |
| D.3 | Apply **cosine similarity** (search, near-duplicates and label noise, class and game similarity) |
| D.4 | Compare at least three classification models (+ baseline), tuning, overfitting check (validation and learning curves), final model, test once (D.4.5) |
| D.5 | Interpret the model (most indicative words) |
| D.6 | **Analyse incorrectly classified texts** — and, for section 8, the successful predictions |
| D.7 | Answer RQ1, conclusions, limitations, future improvements |

**Focus:** the final model uses only the **review text** — RQ1 asks how far the words alone go. Section 1.5 says tabular features known at review time are a *secondary* input; their contribution on top of the text is measured in notebook 03 and checked again here in D.4.4 (validation only), so the text-only choice is justified by numbers.

**Rules:** everything learned (vocabulary, IDF weights, models) is fitted on train; every choice (representation, hyper-parameters, model, threshold) is made on validation; the test set is used **once**, in D.4.5.

**How to read the markdown cells.** The interpretation of the results is printed by the code (cells marked *Findings*), so the text always matches the numbers of the current run. Copy those lines into the report.

Runtime on the full sample: roughly 20–35 minutes (most of it is building 8 different vocabularies in D.2).

## 0. Setup

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC

from src import config as C

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 160)
sns.set_theme(style="whitegrid", context="notebook")
for folder in (C.FIGURES_DIR, C.TABLES_DIR, C.MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RS = C.RANDOM_STATE
NEG, POS = "Not recommended", "Recommended"
PALETTE = {NEG: "#c0392b", POS: "#27ae60"}
TIE_REP = 0.002     # D.2: representations closer than this count as equal -> smaller vocabulary wins
TIE_MODEL = 0.003   # D.4: models closer than this count as equal -> simpler model wins


def savefig(name):
    plt.tight_layout()
    plt.savefig(C.FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def verdict(diff, tol=TIE_REP):
    """Plain-language reading of a difference in validation F1-macro."""
    if diff > tol:
        return "helps"
    if diff < -tol:
        return "hurts"
    return "no real difference"


METRIC_COLS = ["f1_macro", "recall_neg", "precision_neg", "f1_neg", "pr_auc_neg", "roc_auc", "balanced_acc", "accuracy"]


def predict_scores(model, X):
    """Return (score for 'Not recommended', predicted class). Larger score = more likely negative."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        neg_col = list(model.classes_).index(0)
        return proba[:, neg_col], np.asarray(model.classes_)[proba.argmax(axis=1)]
    dec = model.decision_function(X)
    return -dec, np.where(dec > 0, model.classes_[1], model.classes_[0])


def metrics_from_pred(y, pred, score):
    is_neg = (np.asarray(y) == 0)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro", zero_division=0),
        "precision_neg": precision_score(y, pred, pos_label=0, zero_division=0),
        "recall_neg": recall_score(y, pred, pos_label=0, zero_division=0),
        "f1_neg": f1_score(y, pred, pos_label=0, zero_division=0),
        "pr_auc_neg": average_precision_score(is_neg, score),
        "roc_auc": roc_auc_score(is_neg, score) if np.unique(score).size > 1 else 0.5,
    }


def evaluate(model, X, y, threshold=None):
    score, pred = predict_scores(model, X)
    if threshold is not None:
        pred = np.where(score >= threshold, 0, 1)
    return metrics_from_pred(y, pred, score)


def best_threshold(y, score):
    """Threshold on the negative score that maximises F1-macro."""
    candidates = np.unique(np.quantile(score, np.linspace(0.50, 0.995, 200)))
    f1s = [f1_score(y, np.where(score >= t, 0, 1), average="macro") for t in candidates]
    i = int(np.argmax(f1s))
    return float(candidates[i]), float(f1s[i]), candidates, np.array(f1s)


def crossfit_tuned_f1(y, score, seed=RS):
    """Honest 'F1-macro with a tuned threshold' on validation.

    Validation is split into two stratified halves: the threshold is chosen on one half and scored on the other,
    then the halves swap. Every model gets its own best threshold, so models without class weights
    (Naive Bayes) are not penalised by the default threshold, and the score is not optimistic."""
    rng = np.random.default_rng(seed)
    y, score = np.asarray(y), np.asarray(score)
    half_a, half_b = [], []
    for c in (0, 1):
        ids = rng.permutation(np.where(y == c)[0])
        a, b = np.array_split(ids, 2)
        half_a.append(a)
        half_b.append(b)
    half_a, half_b = np.concatenate(half_a), np.concatenate(half_b)
    f1s = []
    for fit_part, score_part in ((half_a, half_b), (half_b, half_a)):
        t = best_threshold(y[fit_part], score[fit_part])[0]
        f1s.append(f1_score(y[score_part], np.where(score[score_part] >= t, 0, 1), average="macro"))
    return float(np.mean(f1s))


def bootstrap_ci(y, pred, score, n_boot=1000, seed=0):
    """95% bootstrap confidence intervals for the main test metrics, including the PAIRED gain over the baseline
    (model and baseline are scored on the same resampled reviews, so the interval of the gain is honest)."""
    rng = np.random.default_rng(seed)
    y, pred, score = np.asarray(y), np.asarray(pred), np.asarray(score)
    rows = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        f1_model = f1_score(y[i], pred[i], average="macro", zero_division=0)
        f1_base = f1_score(y[i], np.ones_like(y[i]), average="macro", zero_division=0)
        ap = average_precision_score(y[i] == 0, score[i])
        rows.append({"f1_macro": f1_model,
                     "recall_neg": recall_score(y[i], pred[i], pos_label=0, zero_division=0),
                     "pr_auc_neg": ap,
                     "f1_macro_gain_vs_baseline": f1_model - f1_base,
                     "pr_auc_gain_vs_no_skill": ap - (y[i] == 0).mean()})
    boot = pd.DataFrame(rows)
    return pd.DataFrame({"2.5%": boot.quantile(0.025), "97.5%": boot.quantile(0.975)})


def train_subsample(n_rows, size=20_000, seed=RS):
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n_rows, size=min(size, n_rows), replace=False))


def plot_confusion(y, pred, title, name):
    cm = confusion_matrix(y, pred, labels=[0, 1])
    share = cm / cm.sum(axis=1, keepdims=True)
    labels = np.array([[f"{c:,}\n({s:.1%} of row)" for c, s in zip(r1, r2)] for r1, r2 in zip(cm, share)])
    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    sns.heatmap(share, annot=labels, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=[NEG, POS], yticklabels=[NEG, POS], ax=ax)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    savefig(name)
    return cm

TEXT, RAW = C.TEXT_CLEAN, "review"
splits = {name: pd.read_parquet(C.DATA_PROCESSED / f"{name}_features.parquet") for name in ("train", "val", "test")}
train, val, test = splits["train"], splits["val"], splits["test"]
y_train, y_val, y_test = (p[C.TARGET].to_numpy().astype(int) for p in (train, val, test))
for p in (train, val, test):
    p["n_words"] = p[TEXT].str.split().str.len()
print({name: len(p) for name, p in splits.items()})
print(f"Negative share: train {(y_train == 0).mean():.2%}, validation {(y_val == 0).mean():.2%}, test {(y_test == 0).mean():.2%}")

## D.1 Text cleaning and preprocessing

### D.1.1 What was already done in notebook 01 (section 3.5)
Lower-case → remove URLs → remove Steam formatting tags (`[h1]`, `[b]`, `[spoiler]` …) → collapse whitespace; empty, broken (> 8,000 characters) and letter-free reviews were removed. The original text is kept in `review` for error analysis.

In [ ]:
changed = train[train[RAW].str.lower().str.strip() != train[TEXT]]
display(changed[[RAW, TEXT]].head(5))
print(f"Reviews changed by cleaning (beyond lower-casing): {len(changed) / len(train):.1%} of the training set")

### D.1.2 Tokenisation
A **token** is the unit that becomes one column of the Bag-of-Words matrix. scikit-learn's default pattern `\b\w\w+\b` keeps tokens of **two or more** characters: numbers such as *10* survive, but one-character tokens (*i*, *a*, *5*) are dropped, and **contractions are cut**: *don't* → *don* (the *t* is dropped), *isn't* → *isn*. The alternative pattern used below keeps the apostrophe inside a word (*don't*, *isn't*, *can't* stay whole) and also keeps one-character tokens. Which one is better is decided by validation in D.2.

In [ ]:
DEFAULT_TOKENS = r"(?u)\b\w\w+\b"
KEEP_CONTRACTIONS = r"(?u)\b\w[\w']*\b"
examples = train.loc[train[TEXT].str.contains("n't", regex=False), TEXT].head(3).tolist() or train[TEXT].head(3).tolist()
for text in examples:
    print("TEXT    :", text[:140])
    print("default :", re.findall(DEFAULT_TOKENS, text)[:22])
    print("keep n't:", re.findall(KEEP_CONTRACTIONS, text)[:22])
    print()

### D.1.3 Stop words
Stop words are very frequent function words (*the, is, and*). Removing them is standard for **topic** analysis, but for **sentiment** it is risky: the standard English stop-word list of scikit-learn contains negations and intensifiers. The table shows how often some of them appear in negative and positive training reviews.

Note: the scikit-learn list contains no contractions, so with the *keep n't* tokenizer the standard list still keeps *don't*, *isn't*; it removes *not, no, never, nothing, cannot* … The custom list keeps those as well.

In [ ]:
SENTIMENT_STOP_WORDS = {"no", "not", "nor", "never", "nothing", "none", "nobody", "noone", "nowhere", "neither",
                        "cannot", "cant", "without", "against", "very", "too", "most", "more", "less", "least",
                        "few", "much", "only", "but", "however", "although", "well", "enough", "again"}
in_list = sorted(SENTIMENT_STOP_WORDS & ENGLISH_STOP_WORDS)
CUSTOM_STOP = sorted(ENGLISH_STOP_WORDS - SENTIMENT_STOP_WORDS)
rows = []
for w in in_list:
    has = train[TEXT].str.contains(rf"\b{w}\b", regex=True).to_numpy()
    rows.append({"word": w, "% of negative reviews": 100 * has[y_train == 0].mean(),
                 "% of positive reviews": 100 * has[y_train == 1].mean()})
stop_table = pd.DataFrame(rows).set_index("word")
stop_table["ratio neg/pos"] = stop_table["% of negative reviews"] / stop_table["% of positive reviews"].clip(lower=0.01)
stop_table = stop_table.sort_values("ratio neg/pos", ascending=False)
display(stop_table.round(2).head(15))
print(f"Standard list: {len(ENGLISH_STOP_WORDS)} words; custom list (sentiment words kept): {len(CUSTOM_STOP)} words")

# Findings
more_neg = stop_table[stop_table["ratio neg/pos"] >= 1.5]
print(f"\nFindings: {len(more_neg)} of the {len(stop_table)} sentiment-bearing stop words are at least 1.5x more frequent "
      f"in negative reviews; the strongest: "
      + ", ".join(f"'{w}' ({r:.1f}x)" for w, r in more_neg["ratio neg/pos"].head(5).items()) + ".")

If negations are clearly more frequent in negative reviews, removing them deletes part of the signal (*"not worth it"* → *"worth"*). Therefore two variants are tested in D.2: the standard list and a **custom list** without negations and intensifiers.

**Stemming / lemmatisation** (*crashes, crashed, crashing → crash*) is not applied:
* with ~130k training reviews every common word form has enough examples on its own, and bigrams already capture most phrases;
* stemmers make mistakes that add noise — the Porter stemmer maps both *universe* and *university* to *univers*, and *was* to *wa*;
* it would be one more choice to tune. It is listed as a future improvement (D.7).

### D.1.4 Vocabulary statistics

In [ ]:
cv_all = CountVectorizer(token_pattern=KEEP_CONTRACTIONS)
counts = cv_all.fit_transform(train[TEXT])
term_freq = np.asarray(counts.sum(axis=0)).ravel()
doc_freq = np.asarray((counts > 0).sum(axis=0)).ravel()
tokens_per_review = np.asarray(counts.sum(axis=1)).ravel()
vocab_all = cv_all.get_feature_names_out()
order = np.argsort(term_freq)[::-1]

vocab_stats = pd.Series({
    "distinct tokens in train": f"{len(vocab_all):,}",
    "tokens in train (total)": f"{int(term_freq.sum()):,}",
    "median tokens per review": f"{np.median(tokens_per_review):.0f}",
    "tokens that occur only once": f"{(term_freq == 1).mean():.1%} of the vocabulary",
    "tokens in >= 5 reviews (min_df = 5)": f"{(doc_freq >= 5).sum():,}",
    "top-100 tokens cover": f"{term_freq[order[:100]].sum() / term_freq.sum():.1%} of all tokens",
}, name="value").to_frame()
display(vocab_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ranks = np.arange(1, len(order) + 1)
axes[0].loglog(ranks, term_freq[order], color="#2c7fb8")
axes[0].set_xlabel("rank of the token (log)")
axes[0].set_ylabel("frequency (log)")
axes[0].set_title("Zipf's law: a few tokens are very frequent, most are rare")
top20 = order[:20]
axes[1].barh(vocab_all[top20][::-1], term_freq[top20][::-1], color="#95a5a6")
axes[1].set_title("20 most frequent tokens")
savefig("D01_vocabulary_zipf")

# Findings
n_stop_top20 = sum(t in ENGLISH_STOP_WORDS for t in vocab_all[top20])
print(f"Findings: {(term_freq == 1).mean():.1%} of distinct tokens occur only once; min_df = 5 keeps "
      f"{(doc_freq >= 5).sum():,} of {len(vocab_all):,} tokens ({(doc_freq >= 5).mean():.1%}). "
      f"{n_stop_top20} of the 20 most frequent tokens are stop words.")
del counts

**Why this matters.** A straight line on the log–log plot (Zipf's law) means the vocabulary has a very long tail of rare words and typos. `min_df = 5` (a term must occur in at least 5 reviews) removes most of that tail: it shrinks the matrix and removes words the model could only memorise. The most frequent tokens carry little sentiment, which is what IDF down-weights automatically.

## D.2 Bag-of-Words vs TF-IDF

* **Bag-of-Words (BoW):** one column per token; the value is the count of the token in the review (or 1/0 = present / absent). Word order is lost — hence "bag".
* **TF-IDF:** the count is replaced by *term frequency × inverse document frequency*. A word that occurs in almost every review (*game*) gets a small weight, a word that is frequent in this review but rare overall (*refund*) gets a large one. With `sublinear_tf` the count is replaced by `1 + log(count)`, so repeating a word ten times does not make it ten times more important. Rows are L2-normalised, so long and short reviews are comparable.
* **n-grams:** bigrams add pairs of neighbouring words (*not worth*, *waste of*, *highly recommend*) and bring back part of the word order.

All variants are scored with the same classifier (class-weighted logistic regression, C = 1), trained on train and evaluated on validation — so the only difference is the representation of the text. **Robustness check:** because the representation is chosen with logistic regression, every variant is also scored with Complement Naive Bayes (`nb_f1_macro`), to show whether the choice depends on the classifier.

In [ ]:
def lr_probe():
    return LogisticRegression(C=1.0, solver="liblinear", class_weight="balanced", max_iter=3000)

base = dict(min_df=5, max_df=0.9)
VECTORIZERS = {
    "BoW binary, unigrams":                       CountVectorizer(binary=True, token_pattern=KEEP_CONTRACTIONS, **base),
    "BoW counts, unigrams":                       CountVectorizer(token_pattern=KEEP_CONTRACTIONS, **base),
    "TF-IDF, unigrams":                           TfidfVectorizer(sublinear_tf=True, token_pattern=KEEP_CONTRACTIONS, **base),
    "TF-IDF, 1-2 grams, 100k (notebook 03)":      TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), max_features=100_000, **base),
    "TF-IDF, 1-2 grams, 100k, keep n't":          TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), max_features=100_000,
                                                                  token_pattern=KEEP_CONTRACTIONS, **base),
    "TF-IDF, 1-2 grams, 100k, standard stop list": TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), max_features=100_000,
                                                                   token_pattern=KEEP_CONTRACTIONS, stop_words="english", **base),
    "TF-IDF, 1-2 grams, 100k, custom stop list":  TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), max_features=100_000,
                                                                  token_pattern=KEEP_CONTRACTIONS, stop_words=CUSTOM_STOP, **base),
    "TF-IDF, 1-2 grams, 300k, keep n't":          TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), max_features=300_000,
                                                                  token_pattern=KEEP_CONTRACTIONS, **base),
}
rep_rows = {}
for name, vec in VECTORIZERS.items():
    t = time.time()
    Xtr = vec.fit_transform(train[TEXT])
    Xva = vec.transform(val[TEXT])
    lr_m = evaluate(lr_probe().fit(Xtr, y_train), Xva, y_val)
    nb_m = evaluate(ComplementNB(alpha=1.0).fit(Xtr, y_train), Xva, y_val)
    rep_rows[name] = {"features": Xtr.shape[1], "seconds": round(time.time() - t), **lr_m,
                      "nb_f1_macro": nb_m["f1_macro"], "nb_pr_auc_neg": nb_m["pr_auc_neg"]}
    print(f"{name:<46} {Xtr.shape[1]:>8,} features  LR val F1-macro {lr_m['f1_macro']:.4f}  PR-AUC {lr_m['pr_auc_neg']:.4f}  "
          f"| NB val F1-macro {nb_m['f1_macro']:.4f}")
representation_table = pd.DataFrame(rep_rows).T
representation_table.to_csv(C.TABLES_DIR / "D_representation_comparison.csv")
del Xtr, Xva

In [ ]:
display(representation_table[["features", "seconds", "f1_macro", "recall_neg", "precision_neg", "pr_auc_neg",
                              "nb_f1_macro"]].astype(float).round(4))
fig, ax = plt.subplots(figsize=(10, 4.8))
rt = representation_table["f1_macro"].astype(float)
best_rep = rt.idxmax()
ax.barh(rt.index[::-1], rt.values[::-1], color=["#c0392b" if n == best_rep else "#95a5a6" for n in rt.index[::-1]])
ax.set_xlim(max(0, rt.min() - 0.03), min(1, rt.max() + 0.01))
ax.set_xlabel("validation F1-macro (logistic regression, C = 1)")
ax.set_title("Text representations compared")
savefig("D02_representation_comparison")

# a smaller vocabulary wins ties: the difference must be at least TIE_REP F1-macro
feat = representation_table["features"].astype(float)
close = rt[rt >= rt.max() - TIE_REP].index
BEST_REP = feat[close].idxmin()
print(f"Best F1-macro: {best_rep} ({rt.max():.4f}); chosen (smallest vocabulary within {TIE_REP}): {BEST_REP} ({rt[BEST_REP]:.4f})")

**Findings.** The next cell turns the table into statements (difference in validation F1-macro, logistic regression; "no real difference" = within ±0.002). Use these lines in the report instead of general expectations — on this data some of the textbook expectations may not hold.

In [ ]:
def rep_f1(name, col="f1_macro"):
    return float(representation_table.loc[name, col])

REP_CHECKS = [
    ("TF-IDF instead of raw counts (unigrams)", "TF-IDF, unigrams", "BoW counts, unigrams"),
    ("binary presence instead of counts (unigrams)", "BoW binary, unigrams", "BoW counts, unigrams"),
    ("adding bigrams to TF-IDF", "TF-IDF, 1-2 grams, 100k, keep n't", "TF-IDF, unigrams"),
    ("keeping contractions (don't) vs default tokens", "TF-IDF, 1-2 grams, 100k, keep n't", "TF-IDF, 1-2 grams, 100k (notebook 03)"),
    ("standard stop list vs no stop list", "TF-IDF, 1-2 grams, 100k, standard stop list", "TF-IDF, 1-2 grams, 100k, keep n't"),
    ("custom stop list (negations kept) vs no stop list", "TF-IDF, 1-2 grams, 100k, custom stop list", "TF-IDF, 1-2 grams, 100k, keep n't"),
    ("custom stop list vs standard stop list", "TF-IDF, 1-2 grams, 100k, custom stop list", "TF-IDF, 1-2 grams, 100k, standard stop list"),
    ("300k instead of 100k vocabulary", "TF-IDF, 1-2 grams, 300k, keep n't", "TF-IDF, 1-2 grams, 100k, keep n't"),
]
rep_findings = pd.DataFrame([{"comparison": label, "A": a, "B": b,
                              "F1 A − B (LR)": rep_f1(a) - rep_f1(b),
                              "verdict (LR)": verdict(rep_f1(a) - rep_f1(b)),
                              "F1 A − B (NB)": rep_f1(a, "nb_f1_macro") - rep_f1(b, "nb_f1_macro"),
                              "verdict (NB)": verdict(rep_f1(a, "nb_f1_macro") - rep_f1(b, "nb_f1_macro"))}
                             for label, a, b in REP_CHECKS]).set_index("comparison")
rep_findings.to_csv(C.TABLES_DIR / "D_representation_findings.csv")
display(rep_findings.round(4))
for label, r in rep_findings.iterrows():
    print(f"- {label}: {r['verdict (LR)']} ({r['F1 A − B (LR)']:+.4f} F1-macro)")

nb_rank = representation_table["nb_f1_macro"].astype(float).rank(ascending=False)
agree = int(nb_rank[BEST_REP])
print(f"\nRobustness: the chosen representation is ranked #{agree} of {len(nb_rank)} by Naive Bayes "
      f"(NB's best: {representation_table['nb_f1_macro'].astype(float).idxmax()}). "
      + ("The choice does not depend on the classifier." if agree <= 2 else
         "Naive Bayes prefers another representation — mention this as a limitation (the choice was made with LR)."))

The chosen vectorizer is refitted once on train and used in the rest of the notebook.

In [ ]:
vectorizer = VECTORIZERS[BEST_REP]
X_train = vectorizer.fit_transform(train[TEXT]).tocsr()
X_val, X_test = vectorizer.transform(val[TEXT]).tocsr(), vectorizer.transform(test[TEXT]).tocsr()
terms = np.asarray(vectorizer.get_feature_names_out())
IS_TFIDF = isinstance(vectorizer, TfidfVectorizer)
print(f"{BEST_REP}: train {X_train.shape}, density {X_train.nnz / np.prod(X_train.shape):.5f}")

## D.3 Cosine similarity

Cosine similarity measures the angle between two vectors: 1 = the reviews use the same words in the same proportions, 0 = no word in common. It ignores length, so a short and a long review about the same thing can still be similar. After L2-normalisation (TF-IDF rows already have length 1; BoW rows are normalised here) the cosine similarity of two reviews is simply their **dot product** — a sparse matrix multiplication computes millions of similarities at once.

Four uses in this project:
1. **search** — find the reviews most similar to a query (how a support team would look for all reports of one problem);
2. **near-duplicates** between validation and train — are there "copies" that make the evaluation too easy, and do identical texts always have the same label?
3. **class centroids** — how far apart are the "average negative" and the "average positive" review? This is also a classifier (D.4);
4. **game similarity** — which games are described with the same words.

In [ ]:
Xn_train, Xn_val = normalize(X_train), normalize(X_val)


def most_similar(query_vec, k=3):
    sims = (Xn_train @ normalize(query_vec).T).toarray().ravel()
    top = np.argsort(sims)[::-1][:k]
    return top, sims[top]


queries = ["the game keeps crashing after the update", "pay to win microtransactions everywhere",
           "full of cheaters and hackers", "beautiful story and amazing soundtrack"]
search_rows = []
for q in queries:
    idx, sims = most_similar(vectorizer.transform([q]))
    for i, s in zip(idx, sims):
        search_rows.append({"query": q, "cosine": round(float(s), 3), "label": POS if y_train[i] == 1 else NEG,
                            "game": train["app_name"].iloc[i], "review": train[RAW].iloc[i][:180]})
display(pd.DataFrame(search_rows).set_index(["query", "cosine"]))

In [ ]:
# Near-duplicates: maximum similarity of each validation review to ANY training review (random 1,000 reviews)
rng = np.random.default_rng(RS)
sample_val = np.sort(rng.choice(X_val.shape[0], size=min(1000, X_val.shape[0]), replace=False))
max_sim, nn_idx = [], []
for start in range(0, len(sample_val), 100):
    block = (Xn_val[sample_val[start:start + 100]] @ Xn_train.T).tocsr()
    max_sim.append(block.max(axis=1).toarray().ravel())
    nn_idx.append(np.asarray(block.argmax(axis=1)).ravel())
max_sim, nn_idx = np.concatenate(max_sim), np.concatenate(nn_idx)
same_label = y_val[sample_val] == y_train[nn_idx]

dup = pd.DataFrame({"max cosine": max_sim, "nearest has same label": same_label,
                    "words": val["n_words"].to_numpy()[sample_val]})
# upper edge slightly above 1: a dot product of two unit vectors can be 1.0000001 because of rounding
dup["similarity band"] = pd.cut(dup["max cosine"], [-0.01, 0.3, 0.5, 0.7, 0.9, 0.999, 1.001],
                                labels=["< 0.3", "0.3–0.5", "0.5–0.7", "0.7–0.9", "0.9–0.999", "identical"])
assert dup["similarity band"].notna().all(), "a similarity fell outside the bands"
band_table = dup.groupby("similarity band", observed=False).agg(
    reviews=("max cosine", "size"), median_words=("words", "median"), same_label_share=("nearest has same label", "mean"))
band_table["share_%"] = 100 * band_table["reviews"] / len(dup)
display(band_table.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(max_sim, bins=40, color="#2c7fb8")
ax.set_xlabel("cosine similarity to the closest training review")
ax.set_ylabel("validation reviews")
ax.set_title("How similar are validation reviews to the training set?")
savefig("D03_nearest_neighbour_similarity")

identical = dup.index[dup["max cosine"] > 0.999]
ex = pd.DataFrame({"validation review": val[RAW].to_numpy()[sample_val[identical[:6]]],
                   "label": np.where(y_val[sample_val[identical[:6]]] == 1, POS, NEG),
                   "closest train label": np.where(y_train[nn_idx[identical[:6]]] == 1, POS, NEG)})
display(ex)

In [ ]:
# Label noise: the same text written by many different players — how often do they disagree about the thumb?
text_groups = pd.DataFrame({"text": train[TEXT].to_numpy(), "y": y_train}).groupby("text")["y"].agg(["size", "mean"])
frequent = text_groups[text_groups["size"] >= 20].sort_values("size", ascending=False)
frequent = frequent.assign(negative_share=1 - frequent["mean"]).drop(columns="mean")
minority = np.minimum(frequent["negative_share"], 1 - frequent["negative_share"])
noise_rate = float((minority * frequent["size"]).sum() / frequent["size"].sum()) if len(frequent) else float("nan")
display(frequent.head(12).rename(columns={"size": "training reviews"}).round(3))
frequent.to_csv(C.TABLES_DIR / "D_frequent_identical_texts.csv")

# Findings
n_ident = len(identical)
print(f"Findings: {n_ident / len(dup):.1%} of sampled validation reviews have an identical text in train "
      f"(median length {dup.loc[identical, 'words'].median() if n_ident else 0:.0f} words); "
      f"for {100 * (1 - dup.loc[identical, 'nearest has same label'].mean()) if n_ident else 0:.1f}% of them the identical "
      f"training review has the OPPOSITE label.")
print(f"Among the {len(frequent)} texts written at least 20 times in train ({int(frequent['size'].sum()):,} reviews), "
      f"{100 * noise_rate:.1f}% of the reviews carry the minority label of their text. "
      "No text-only model can classify those correctly: this is a lower bound on the error for such reviews.")

**How to read it.** Since exact duplicate rows and repeated (author, game) pairs were removed in notebook 01 (3.3), identical texts here are reviews written by **different players** — typically short generic texts. They are legitimate and stay in the data. The important number is how often identical texts carry **opposite** labels (joke, sarcasm, mis-click): the model sees the same input with two different answers, so part of those reviews will always be misclassified. This sets a ceiling on accuracy that is discussed again in D.6.

In [ ]:
# Class centroids
centroid = {c: normalize(np.asarray(Xn_train[y_train == c].mean(axis=0))) for c in (0, 1)}
centroid_cos = float((centroid[0] @ centroid[1].T).item())
sim_neg = np.asarray(Xn_val @ centroid[0].T).ravel()
sim_pos = np.asarray(Xn_val @ centroid[1].T).ravel()
diff = sim_neg - sim_pos

fig, ax = plt.subplots(figsize=(8, 4))
for c, lab in ((1, POS), (0, NEG)):
    ax.hist(diff[y_val == c], bins=60, alpha=0.6, density=True, color=PALETTE[lab], label=lab)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("cos(review, negative centroid) − cos(review, positive centroid)")
ax.set_title(f"Validation reviews vs class centroids (centroid similarity = {centroid_cos:.3f})")
ax.legend()
savefig("D04_centroid_similarity")

# Findings
closer_own_neg = (diff[y_val == 0] > 0).mean()
closer_own_pos = (diff[y_val == 1] < 0).mean()
print(f"Findings: cosine similarity between the average negative and the average positive review = {centroid_cos:.3f}. "
      f"{closer_own_neg:.1%} of negative and {closer_own_pos:.1%} of positive validation reviews are closer to the "
      f"centroid of their own class (median difference: negative {np.median(diff[y_val == 0]):+.3f}, "
      f"positive {np.median(diff[y_val == 1]):+.3f}).")

A high centroid similarity means both classes use mostly the same vocabulary (the same games, the same words). The question is whether the two histograms are shifted apart: if each class is mostly closer to its own centroid, the rule "which centroid is closer" already works as a classifier — the **nearest-centroid (Rocchio) classifier**, one of the models in D.4.

In [ ]:
# Game similarity: average review vector of each of the 12 largest games
top_games = train["app_name"].value_counts().index[:12]
G = normalize(np.vstack([np.asarray(Xn_train[(train["app_name"] == g).to_numpy()].mean(axis=0)) for g in top_games]))
game_sim = pd.DataFrame(G @ G.T, index=top_games, columns=top_games)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(game_sim, annot=True, fmt=".2f", cmap="viridis", annot_kws={"size": 7}, square=True, ax=ax)
ax.set_title("Cosine similarity between games (average review vector)")
savefig("D05_game_similarity")
pairs = game_sim.where(np.triu(np.ones(game_sim.shape, dtype=bool), k=1)).stack().dropna().sort_values(ascending=False)
print("Most similar pairs:", "; ".join(f"{a} – {b} ({s:.2f})" for (a, b), s in pairs.head(3).items()))
print("Least similar pairs:", "; ".join(f"{a} – {b} ({s:.2f})" for (a, b), s in pairs.tail(3).items()))

## D.4 Classification models

| Model | Idea | Tuned |
|---|---|---|
| Baseline | always 👍 | – |
| **Nearest centroid (cosine)** | closer to the average 👎 review or to the average 👍 review? | – (no hyper-parameters) |
| **Complement Naive Bayes** | probability of each word in each class, words assumed independent; the *complement* variant was designed for imbalanced text data | `alpha` (smoothing) |
| **Logistic regression** | weighted sum of word weights → probability | `C` |
| **Linear SVM** | maximum-margin separation in the word space | `C` |

**Class imbalance** (≈ 8 positive reviews per negative one) is handled in two ways:
1. `class_weight="balanced"` for LR and SVM: an error on a 👎 review costs ~8× more. In expectation this is equivalent to oversampling the rare class, without copying or throwing away reviews; validation and test keep the real class ratio.
2. **Threshold moving.** Naive Bayes and the centroid classifier have no class weights, so their default threshold is not comparable. To compare all models fairly, every model is also scored with its **own tuned threshold**: `f1_macro_tuned` = threshold chosen on one half of validation and scored on the other half, then swapped (cross-fitting, so the number is not optimistic). **Hyper-parameters and the final model are chosen by `f1_macro_tuned`**, because the final model is used with a tuned threshold. `f1_macro` at the default threshold and the threshold-free **PR-AUC** are reported next to it.

Every model is also scored on a 20,000-row subsample of train to check overfitting.

In [ ]:
class CosineCentroidClassifier(ClassifierMixin, BaseEstimator):
    """Assigns a review to the class whose average (centroid) vector has the higher cosine similarity."""

    def fit(self, X, y):
        Xn = normalize(X)
        y = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.centroids_ = normalize(np.vstack([np.asarray(Xn[y == c].mean(axis=0)) for c in self.classes_]))
        return self

    def decision_function(self, X):
        sims = np.asarray(normalize(X) @ self.centroids_.T)
        return sims[:, 1] - sims[:, 0]

    def predict(self, X):
        return np.where(self.decision_function(X) > 0, 1, 0)


SELECT_BY = "f1_macro_tuned"
RESULTS, FITTED, VAL_SCORES = [], {}, {}
sub = train_subsample(X_train.shape[0])


def tune(name, make_model, grid):
    best = None
    for params in grid:
        model = make_model(**params)
        t = time.time()
        model.fit(X_train, y_train)
        fit_s = time.time() - t
        score_val, _ = predict_scores(model, X_val)
        val_m = evaluate(model, X_val, y_val)
        val_m["f1_macro_tuned"] = crossfit_tuned_f1(y_val, score_val)
        tr_m = evaluate(model, X_train[sub], y_train[sub])
        RESULTS.append({"model": name, "params": json.dumps(params), "fit_s": round(fit_s, 1),
                        "train_f1_macro": tr_m["f1_macro"], **val_m})
        print(f"{name:<20} {json.dumps(params):<16} val F1 {val_m['f1_macro']:.4f}  tuned {val_m['f1_macro_tuned']:.4f}  "
              f"PR-AUC {val_m['pr_auc_neg']:.4f}  train F1 {tr_m['f1_macro']:.4f}  ({fit_s:.0f} s)")
        if best is None or val_m[SELECT_BY] > best["val"][SELECT_BY]:
            best = {"model": model, "params": params, "val": val_m, "train": tr_m}
    FITTED[name] = best
    VAL_SCORES[name] = predict_scores(best["model"], X_val)[0]


majority = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_val = evaluate(majority, X_val, y_val)
baseline_val["f1_macro_tuned"] = baseline_val["f1_macro"]   # constant score: no threshold to tune
GRIDS = {"Complement NB": (0.1, 0.3, 1.0, 3.0), "Logistic regression": (0.3, 1.0, 3.0, 10.0), "Linear SVM": (0.03, 0.1, 0.3, 1.0)}
tune("Nearest centroid", lambda: CosineCentroidClassifier(), [{}])
tune("Complement NB", lambda alpha: ComplementNB(alpha=alpha), [{"alpha": a} for a in GRIDS["Complement NB"]])
tune("Logistic regression", lambda C_: LogisticRegression(C=C_, solver="liblinear", class_weight="balanced", max_iter=3000),
     [{"C_": c} for c in GRIDS["Logistic regression"]])
tune("Linear SVM", lambda C_: LinearSVC(C=C_, class_weight="balanced", max_iter=5000, random_state=RS),
     [{"C_": c} for c in GRIDS["Linear SVM"]])
pd.DataFrame(RESULTS).to_csv(C.TABLES_DIR / "D_tuning_all_versions.csv", index=False)

### D.4.1 Comparison on validation and overfitting check

In [ ]:
comparison = pd.DataFrame({"Baseline: majority": {"params": "-", "train_f1_macro": np.nan, **baseline_val},
                           **{n: {"params": json.dumps(f["params"]), "train_f1_macro": f["train"]["f1_macro"], **f["val"]}
                              for n, f in FITTED.items()}}).T
num_cols = METRIC_COLS + ["f1_macro_tuned", "train_f1_macro"]
comparison[num_cols] = comparison[num_cols].astype(float)
comparison["overfit gap"] = comparison["train_f1_macro"] - comparison["f1_macro"]
comparison = comparison.sort_values(SELECT_BY, ascending=False)
comparison.to_csv(C.TABLES_DIR / "D_model_comparison_validation.csv")
display(comparison[["params", "f1_macro_tuned", "f1_macro", "recall_neg", "precision_neg", "pr_auc_neg", "roc_auc",
                    "accuracy", "train_f1_macro", "overfit gap"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
comparison[["f1_macro_tuned", "f1_macro", "recall_neg", "precision_neg", "pr_auc_neg"]].iloc[::-1].plot(
    kind="barh", ax=axes[0], width=0.85)
axes[0].set_xlim(0, 1)
axes[0].set_title("Text models — validation")
axes[0].legend(fontsize=8, loc="lower right")
for name, score in VAL_SCORES.items():
    p, r, _ = precision_recall_curve(y_val == 0, score)
    axes[1].plot(r, p, label=f"{name} (AP {average_precision_score(y_val == 0, score):.3f})")
axes[1].axhline((y_val == 0).mean(), ls="--", color="grey", label="baseline (no skill)")
axes[1].set_xlabel("recall (Not recommended)")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision–recall curves — validation")
axes[1].legend(fontsize=8)
savefig("D06_text_model_comparison")

tuning = pd.DataFrame(RESULTS)
param_key = {"Complement NB": "alpha", "Logistic regression": "C_", "Linear SVM": "C_"}
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, name in zip(axes, ["Complement NB", "Logistic regression", "Linear SVM"]):
    part = tuning[tuning["model"] == name]
    xs = [json.loads(p)[param_key[name]] for p in part["params"]]
    ax.semilogx(xs, part["train_f1_macro"], marker="o", label="train (subsample)")
    ax.semilogx(xs, part["f1_macro"], marker="s", label="validation, default threshold")
    ax.semilogx(xs, part["f1_macro_tuned"], marker="^", label="validation, tuned threshold")
    ax.set_xlabel("alpha (more smoothing →)" if name == "Complement NB" else "C (weaker regularisation →)")
    ax.set_ylabel("F1-macro")
    ax.set_title(f"Validation curve — {name}")
    ax.legend(fontsize=8)
savefig("D07_validation_curves")

**Findings: overfitting.** A gap between train and validation F1 is expected for text models with many features: some rare words are seen in only a handful of training reviews. The cell below classifies each gap (< 0.02 small, 0.02–0.05 moderate, > 0.05 large — declared in advance) and warns when the best hyper-parameter is at the edge of its grid (then the grid should be extended).

In [ ]:
def gap_label(g):
    return "small" if g < 0.02 else ("moderate" if g <= 0.05 else "large")

for name, f in FITTED.items():
    g = f["train"]["f1_macro"] - f["val"]["f1_macro"]
    line = f"- {name}: train {f['train']['f1_macro']:.4f} vs validation {f['val']['f1_macro']:.4f} -> gap {g:+.4f} ({gap_label(g)})"
    if name in GRIDS:
        best_p = f["params"][param_key[name]]
        grid = GRIDS[name]
        where = "at the EDGE of the grid — consider extending it" if best_p in (min(grid), max(grid)) else "inside the grid"
        line += f"; best {param_key[name].rstrip('_')} = {best_p} ({where})"
    print(line)

### D.4.2 Final text model
**Rule (declared before looking at the results):** highest validation `f1_macro_tuned`; models within 0.003 of the best are considered equal and the **simpler** one is taken (order: nearest centroid → Naive Bayes → logistic regression → linear SVM; logistic regression is also preferred over SVM because it outputs probabilities). The cell prints explicitly when a simpler model is chosen over a slightly better one — be ready to defend this at the defense: a difference of less than 0.003 F1 is smaller than the noise between two random validation sets, while a simpler model is faster, easier to explain and less likely to overfit.

Then the decision threshold of the final model is tuned on the whole validation set.

In [ ]:
order_simple = ["Nearest centroid", "Complement NB", "Logistic regression", "Linear SVM"]
best_name = max(FITTED, key=lambda n: FITTED[n]["val"][SELECT_BY])
best_f1 = FITTED[best_name]["val"][SELECT_BY]
FINAL_NAME = next(n for n in order_simple if FITTED[n]["val"][SELECT_BY] >= best_f1 - TIE_MODEL)
final_model = FITTED[FINAL_NAME]["model"]
if FINAL_NAME != best_name:
    print(f"Note: {best_name} has the highest validation F1-macro (tuned) {best_f1:.4f}; {FINAL_NAME} is "
          f"{best_f1 - FITTED[FINAL_NAME]['val'][SELECT_BY]:.4f} lower — within the tolerance {TIE_MODEL}, "
          "so the simpler model is kept.")

val_score, _ = predict_scores(final_model, X_val)
thr, _, thr_grid, thr_f1s = best_threshold(y_val, val_score)
tuned_val = evaluate(final_model, X_val, y_val, threshold=thr)
print(f"Final text model: {FINAL_NAME} {FITTED[FINAL_NAME]['params']}")
print(f"  validation F1-macro: default threshold {FITTED[FINAL_NAME]['val']['f1_macro']:.4f}; "
      f"tuned threshold (cross-fitted, honest) {FITTED[FINAL_NAME]['val']['f1_macro_tuned']:.4f}; "
      f"threshold {thr:.3f} chosen on all of validation -> {tuned_val['f1_macro']:.4f} (optimistic: same data)")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(thr_grid, thr_f1s, color="#2c7fb8")
ax.axvline(thr, color="#c0392b", ls="--", label=f"chosen threshold {thr:.3f}")
ax.set_xlabel("threshold on the 'Not recommended' score")
ax.set_ylabel("validation F1-macro")
ax.set_title(f"Threshold tuning — {FINAL_NAME}")
ax.legend()
savefig("D08_threshold_tuning")

### D.4.3 Learning curve
A learning curve trains the final model on 10 %, 25 %, 50 % and 100 % of the training set (same vocabulary, same hyper-parameters) and scores it on train and validation. It answers two questions for the report: is the model overfitting (large, stable gap), and would **more data** help (validation still rising at 100 %)? The second answer supports the "larger sample" improvement in D.7.

In [ ]:
lc_rows = []
rng_lc = np.random.default_rng(RS)
for frac in (0.10, 0.25, 0.50, 1.00):
    n = int(frac * X_train.shape[0])
    idx = np.sort(rng_lc.choice(X_train.shape[0], size=n, replace=False)) if frac < 1 else np.arange(X_train.shape[0])
    m = clone(final_model).fit(X_train[idx], y_train[idx])
    tr_idx = idx if len(idx) <= 20_000 else np.sort(rng_lc.choice(idx, size=20_000, replace=False))
    va = evaluate(m, X_val, y_val)
    lc_rows.append({"train share": frac, "train reviews": n,
                    "train_f1_macro": evaluate(m, X_train[tr_idx], y_train[tr_idx])["f1_macro"],
                    "val_f1_macro": va["f1_macro"], "val_pr_auc_neg": va["pr_auc_neg"]})
learning = pd.DataFrame(lc_rows)
learning.to_csv(C.TABLES_DIR / "D_learning_curve.csv", index=False)
display(learning.round(4))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(learning["train reviews"], learning["train_f1_macro"], marker="o", label="train")
ax.plot(learning["train reviews"], learning["val_f1_macro"], marker="s", label="validation")
ax.set_xlabel("training reviews")
ax.set_ylabel("F1-macro (default threshold)")
ax.set_title(f"Learning curve — {FINAL_NAME}")
ax.legend()
savefig("D09_learning_curve")

# Findings
last_gain = learning["val_f1_macro"].iloc[-1] - learning["val_f1_macro"].iloc[-2]
final_gap = learning["train_f1_macro"].iloc[-1] - learning["val_f1_macro"].iloc[-1]
print(f"Findings: doubling the data from 50% to 100% changed validation F1-macro by {last_gain:+.4f} "
      + ("-> the curve is still rising, more data is likely to help." if last_gain > TIE_REP else
         "-> the curve has flattened; more data of the same kind would help only a little.")
      + f" Train–validation gap at 100%: {final_gap:+.4f} ({gap_label(final_gap)}).")

### D.4.4 Does tabular information add anything? (validation only)
Section 1.5 lists tabular features known at review time (playtime, number of games/reviews of the author, purchase flags, game) as a *secondary* input. This check adds the tabular matrix built by the preprocessor of notebook 01 (fitted on train) to the chosen text matrix and compares logistic regression with and without it, on **validation**. It does not change the final model or touch the test set; it only quantifies the claim "the words alone are enough for RQ1". If the preprocessor file is missing, the cell is skipped and the comparison in notebook 03 is the reference.

In [ ]:
tab_result = None
try:
    prep = joblib.load(C.MODELS_DIR / "preprocessor_tabular.joblib")

    def as_csr(M):
        return M.tocsr().astype(float) if sparse.issparse(M) else sparse.csr_matrix(np.asarray(M, dtype=float))
    T_tr, T_va = as_csr(prep.transform(train)), as_csr(prep.transform(val))
    lr_c = FITTED["Logistic regression"]["params"]["C_"]
    make_lr = lambda: LogisticRegression(C=lr_c, solver="liblinear", class_weight="balanced", max_iter=3000)
    variants = {"text only": (X_train, X_val), "text + tabular": (sparse.hstack([X_train, T_tr]).tocsr(),
                                                                   sparse.hstack([X_val, T_va]).tocsr()),
                "tabular only": (T_tr, T_va)}
    tab_rows = {}
    for name, (A_tr, A_va) in variants.items():
        m = make_lr().fit(A_tr, y_train)
        s, _ = predict_scores(m, A_va)
        tab_rows[name] = {**evaluate(m, A_va, y_val), "f1_macro_tuned": crossfit_tuned_f1(y_val, s)}
    tab_result = pd.DataFrame(tab_rows).T[["f1_macro_tuned", "f1_macro", "pr_auc_neg", "recall_neg", "precision_neg"]]
    tab_result.to_csv(C.TABLES_DIR / "D_text_vs_tabular_validation.csv")
    display(tab_result.round(4))
    gain = tab_result.loc["text + tabular", "f1_macro_tuned"] - tab_result.loc["text only", "f1_macro_tuned"]
    print(f"Findings: adding the tabular features changes validation F1-macro (tuned) by {gain:+.4f} -> "
          f"{verdict(gain)}. Tabular features alone: {tab_result.loc['tabular only', 'f1_macro_tuned']:.4f} "
          f"vs text alone {tab_result.loc['text only', 'f1_macro_tuned']:.4f}.")
except (FileNotFoundError, KeyError, ValueError) as e:
    print(f"Skipped ({type(e).__name__}: {e}). Use the feature-group comparison of notebook 03 instead.")

### D.4.5 Test set — used once

In [ ]:
test_score, test_pred_default = predict_scores(final_model, X_test)
test_pred = np.where(test_score >= thr, 0, 1)
test_metrics = metrics_from_pred(y_test, test_pred, test_score)
baseline_test = metrics_from_pred(y_test, np.ones_like(y_test), np.zeros(len(y_test)))
test_table = pd.DataFrame({"majority baseline": baseline_test,
                           f"{FINAL_NAME} (default threshold)": metrics_from_pred(y_test, test_pred_default, test_score),
                           f"{FINAL_NAME} (tuned threshold)": test_metrics,
                           f"{FINAL_NAME} (tuned, validation)": tuned_val}).T[METRIC_COLS]
test_table.to_csv(C.TABLES_DIR / "D_final_test_results.csv")
display(test_table.round(4))
ci = bootstrap_ci(y_test, test_pred, test_score)
ci.to_csv(C.TABLES_DIR / "D_test_bootstrap_ci.csv")
display(ci.round(4).rename_axis("95% bootstrap CI (test)"))
cm = plot_confusion(y_test, test_pred, f"{FINAL_NAME} (text only) — test set", "D10_confusion_matrix_test")

# Findings
drop = tuned_val["f1_macro"] - test_metrics["f1_macro"]
print(f"Findings: test F1-macro {test_metrics['f1_macro']:.4f} vs validation {tuned_val['f1_macro']:.4f} "
      f"(test is {'lower' if drop > 0 else 'higher'} by {abs(drop):.4f}; the honest cross-fitted validation estimate was "
      f"{FITTED[FINAL_NAME]['val']['f1_macro_tuned']:.4f}). The model finds {test_metrics['recall_neg']:.1%} of negative "
      f"reviews; {test_metrics['precision_neg']:.1%} of the reviews it flags are really negative.")

## D.5 Interpretation: which words decide?
For every model a "word weight" is defined so that a positive weight pushes towards 👍 and a negative one towards 👎: the coefficient for LR/SVM, the difference of the (complement) log-probabilities for Naive Bayes, the difference of the centroid values for the centroid classifier. For all four models the decision is a weighted sum of the feature values, so these weights are directly comparable within one model.

In [ ]:
def word_weights(model):
    if hasattr(model, "coef_"):
        return np.asarray(model.coef_).ravel()
    if hasattr(model, "feature_log_prob_"):
        return model.feature_log_prob_[1] - model.feature_log_prob_[0]
    return model.centroids_[1] - model.centroids_[0]


weights = pd.Series(word_weights(final_model), index=terms)
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
top_n, top_p = weights.nsmallest(25), weights.nlargest(25)
axes[0].barh(top_n.index[::-1], top_n.values[::-1], color=PALETTE[NEG])
axes[0].set_title("Strongest Not recommended terms")
axes[1].barh(top_p.index[::-1], top_p.values[::-1], color=PALETTE[POS])
axes[1].set_title("Strongest Recommended terms")
savefig("D11_top_terms_final_model")
weights.sort_values().rename("weight").to_csv(C.TABLES_DIR / "D_term_weights.csv")

# Findings
has_bigrams = any(" " in t for t in terms)
if has_bigrams:
    bigrams_neg = [t for t in weights.nsmallest(300).index if " " in t][:15]
    bigrams_pos = [t for t in weights.nlargest(300).index if " " in t][:15]
    print("Strongest negative bigrams:", ", ".join(bigrams_neg))
    print("Strongest positive bigrams:", ", ".join(bigrams_pos))
else:
    print("The chosen representation has no bigrams (unigrams only).")
neg_top50 = list(weights.nsmallest(50).index)
negation_terms = [t for t in neg_top50 if re.search(r"\b(?:not|no|never|nothing)\b|n't", t)]
print(f"Findings: {len(negation_terms)} of the 50 strongest negative terms contain a negation"
      + (f" ({', '.join(negation_terms[:8])})" if negation_terms else "")
      + " — this is the signal the standard stop-word list would remove.")

## D.6 Error analysis: incorrectly (and correctly) classified reviews

The analysis uses the test predictions of the final model. It does not influence the model (it is already fixed); its purpose is to understand the limits of the approach and to plan improvements.

Two kinds of errors:
* **missed negative** — the author gave 👎, the model said 👍 (false negative for the class we are hunting);
* **false alarm** — the author gave 👍, the model said 👎.

### D.6.1 How many errors, and the most confident ones

In [ ]:
err = test[[RAW, TEXT, "app_name", "author_playtime_at_review", "n_words"]].copy().reset_index(drop=True)
err["true"] = y_test
err["pred"] = test_pred
err["neg_score"] = test_score
err["type"] = np.select([(err["true"] == 0) & (err["pred"] == 1), (err["true"] == 1) & (err["pred"] == 0)],
                        ["missed negative", "false alarm"], "correct")
display(err["type"].value_counts().rename("reviews").to_frame()
        .assign(share_pct=lambda d: (100 * d["reviews"] / len(err)).round(2)))

show = [RAW, "app_name", "n_words", "neg_score"]
print("Most confident MISSED NEGATIVES (author: 👎, model very sure it is 👍):")
display(err[err["type"] == "missed negative"].nsmallest(10, "neg_score")[show])
print("Most confident FALSE ALARMS (author: 👍, model very sure it is 👎):")
display(err[err["type"] == "false alarm"].nlargest(10, "neg_score")[show])
err.loc[err["type"] != "correct", show + ["type"]].to_csv(C.TABLES_DIR / "D_misclassified_test_reviews.csv", index=False)

Typical reasons why a Bag-of-Words model fails, known from work on review sentiment. They are **hypotheses**: D.6.3 measures how often each occurs among the errors of this model, and D.6.4 prints **real examples from the test set** for each one — use those (not invented examples) in the presentation.
1. **sarcasm and jokes** — positive words, negative meaning (often with markers such as *10/10*, *lol*);
2. **mixed opinions** — praise and criticism in one review; the decision is often in the part after *but*;
3. **complaints with a 👍** — the author lists problems and still recommends;
4. **too short / no opinion** — one or two words that carry almost no sentiment;
5. **label noise** — the text clearly contradicts the thumb (mis-click, review edited later); identical texts with opposite labels were already found in D.3.

### D.6.2 Errors by review length and by game

In [ ]:
length_bins = pd.cut(err["n_words"], [0, 3, 10, 30, 100, np.inf], labels=["1–3", "4–10", "11–30", "31–100", "> 100"])
by_len = err.assign(error=err["type"] != "correct", length=length_bins).groupby("length", observed=False).agg(
    reviews=("error", "size"), error_rate=("error", "mean"),
    missed_negative=("type", lambda s: (s == "missed negative").mean()),
    false_alarm=("type", lambda s: (s == "false alarm").mean()))
display(by_len.round(4))

big_games = err["app_name"].value_counts().index[:12]
by_game = err[err["app_name"].isin(big_games)].assign(error=lambda d: d["type"] != "correct").groupby("app_name").agg(
    reviews=("error", "size"), error_rate=("error", "mean"), negative_share=("true", lambda s: 1 - s.mean()))
by_game = by_game.sort_values("error_rate", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
by_len[["missed_negative", "false_alarm"]].mul(100).plot(kind="bar", stacked=True, ax=axes[0], rot=0,
                                                       color=["#c0392b", "#f39c12"])
axes[0].set_ylabel("% of reviews in the group")
axes[0].set_xlabel("review length (words)")
axes[0].set_title("Error rate by review length (test)")
axes[1].barh(by_game.index[::-1], 100 * by_game["error_rate"][::-1], color="#7f8c8d")
axes[1].set_xlabel("% misclassified")
axes[1].set_title("Error rate for the 12 largest games (test)")
savefig("D12_errors_by_length_and_game")
display(by_game.round(3))

# Findings
rho_game = by_game["error_rate"].corr(by_game["negative_share"], method="spearman")
print(f"Findings: the error rate is highest for reviews of {by_len['error_rate'].idxmax()} words "
      f"({by_len['error_rate'].max():.1%}) and lowest for {by_len['error_rate'].idxmin()} words "
      f"({by_len['error_rate'].min():.1%}). Across the 12 largest games, error rate and share of negative reviews "
      f"have Spearman rho = {rho_game:+.2f} (games with more negative reviews are "
      f"{'harder' if rho_game > 0.3 else ('easier' if rho_game < -0.3 else 'not clearly harder or easier')}).")

### D.6.3 Error patterns
**Risk ratio** = error rate of reviews with the pattern / error rate of reviews without it. A value > 1 means reviews with this pattern are misclassified more often. This turns the hypotheses above into numbers for the report. The patterns are detected with simple regular expressions, so they are approximate (e.g. not every *10/10* is a joke).

In [ ]:
polar_terms = weights.abs().nlargest(300).index
polar_idx = np.where(np.isin(terms, polar_terms))[0]
has_polar_word = np.asarray((X_test[:, polar_idx] > 0).sum(axis=1)).ravel() > 0

MIXED = "mixed opinion ('but', 'however', ...)"
PATTERNS = {
    "mixed opinion ('but', 'however', ...)": r"\b(?:but|however|although|though|except)\b",
    "joke / sarcasm marker (10/10, lol, /s, meme ...)": r"\b\d+/10\b|\blol\b|\blmao\b|(?:^|\s)/s\b|\bmeme\b|\bjk\b|\bxd\b",
    "negation present": r"\bnot\b|\bno\b|\bnever\b|n't\b",
}
flags = pd.DataFrame({name: err[TEXT].str.contains(rx, regex=True) for name, rx in PATTERNS.items()}, index=err.index)
flags["very short (<= 3 words)"] = err["n_words"] <= 3
flags["no strongly polar word"] = ~has_polar_word
rows = []
for name in flags:
    f = flags[name]
    rows.append({"pattern": name, "% of all test reviews": 100 * f.mean(),
                 "% of missed negatives": 100 * f[err["type"] == "missed negative"].mean(),
                 "% of false alarms": 100 * f[err["type"] == "false alarm"].mean(),
                 "error rate with pattern %": 100 * (err.loc[f, "type"] != "correct").mean() if f.any() else np.nan,
                 "error rate without %": 100 * (err.loc[~f, "type"] != "correct").mean() if (~f).any() else np.nan})
pattern_table = pd.DataFrame(rows).set_index("pattern")
pattern_table["risk ratio"] = (pattern_table["error rate with pattern %"]
                               / pattern_table["error rate without %"].replace(0, np.nan)).replace(np.inf, np.nan)
pattern_table.to_csv(C.TABLES_DIR / "D_error_patterns.csv")
display(pattern_table.round(2))

ax = pattern_table["risk ratio"].dropna().sort_values().plot(kind="barh", figsize=(9, 3.8), color="#8e44ad")
ax.axvline(1, color="black", lw=1)
ax.set_xlabel("error rate with the pattern / error rate without it")
ax.set_title("Which kinds of reviews are hard for the model?")
savefig("D13_error_patterns")

# Findings
for name, r in pattern_table.sort_values("risk ratio", ascending=False).iterrows():
    if pd.notna(r["risk ratio"]):
        print(f"- {name}: in {r['% of all test reviews']:.1f}% of reviews; error rate {r['error rate with pattern %']:.1f}% "
              f"with vs {r['error rate without %']:.1f}% without (risk ratio {r['risk ratio']:.2f})")

### D.6.4 Real examples for each pattern
For every pattern: the three most confident errors that have it. Plus **label-noise candidates**: errors where the model is extremely confident (top 1 % of confidence among errors) — read them and decide whether the text or the thumb is "wrong"; that judgement is the qualitative part of the error analysis.

In [ ]:
err["confidence"] = np.where(err["pred"] == 0, err["neg_score"], -err["neg_score"])   # how sure the model was
wrong = err["type"] != "correct"
example_rows = []
for name in flags:
    part = err[wrong & flags[name]].nlargest(3, "confidence")
    for _, r in part.iterrows():
        example_rows.append({"pattern": name, "type": r["type"], "game": r["app_name"],
                             "review": str(r[RAW])[:220].replace("\n", " ")})
pattern_examples = pd.DataFrame(example_rows)
display(pattern_examples.set_index(["pattern", "type"]))

cut = err.loc[wrong, "confidence"].quantile(0.99) if wrong.any() else np.inf
noise_candidates = err[wrong & (err["confidence"] >= cut)].sort_values("confidence", ascending=False)
print(f"Label-noise candidates (top 1% most confident errors): {len(noise_candidates)} reviews")
display(noise_candidates[[RAW, "app_name", "type", "n_words"]].head(10))
pd.concat([pattern_examples, noise_candidates[[RAW, "app_name", "type"]].rename(columns={RAW: "review", "app_name": "game"})
           .assign(pattern="label-noise candidate")]).to_csv(C.TABLES_DIR / "D_error_examples.csv", index=False)

### D.6.5 Word-by-word explanation of individual errors

In [ ]:
def explain(i, k=6):
    """Contribution of every word = feature value (TF-IDF or count) × word weight (positive -> 👍, negative -> 👎)."""
    row = X_test[i]
    contrib = pd.Series(row.data * weights.values[row.indices], index=terms[row.indices]).sort_values()
    return contrib.head(k), contrib.tail(k)[::-1]


def print_explanation(i, header):
    neg_part, pos_part = explain(i)
    print(f"[{header}] true: {POS if y_test[i] == 1 else NEG} | predicted: {POS if test_pred[i] == 1 else NEG} "
          f"| {err.loc[i, 'app_name']}")
    print("  text:", str(err.loc[i, RAW])[:300].replace("\n", " "))
    print("  pushed to 👎 by:", ", ".join(f"{w} ({v:+.2f})" for w, v in neg_part.items() if v < 0) or "-")
    print("  pushed to 👍 by:", ", ".join(f"{w} ({v:+.2f})" for w, v in pos_part.items() if v > 0) or "-")
    print()


picks = list(err[err["type"] == "missed negative"].nsmallest(2, "neg_score").index) + \
        list(err[err["type"] == "false alarm"].nlargest(2, "neg_score").index)
for i in picks:
    print_explanation(i, err.loc[i, "type"].upper())

### D.6.6 Successful predictions
Section 8 of the assignment asks to discuss successful as well as incorrect predictions. Below: the negative and positive reviews the model classified correctly with the highest confidence, and the word-by-word explanation of one of each. They show what the model has really learned — usually explicit, unambiguous vocabulary.

In [ ]:
correct_neg = err[(err["type"] == "correct") & (err["true"] == 0)].nlargest(5, "neg_score")
correct_pos = err[(err["type"] == "correct") & (err["true"] == 1)].nsmallest(5, "neg_score")
print("Most confident CORRECT negatives:")
display(correct_neg[show])
print("Most confident CORRECT positives:")
display(correct_pos[show])
for i in list(correct_neg.index[:1]) + list(correct_pos.index[:1]):
    print_explanation(i, "CORRECT")

# Findings: what distinguishes correct from wrong predictions
cmp = pd.DataFrame({
    "median words": [err.loc[err["type"] == "correct", "n_words"].median(), err.loc[wrong, "n_words"].median()],
    "% with a strongly polar word": [100 * has_polar_word[(err["type"] == "correct").to_numpy()].mean(),
                                     100 * has_polar_word[wrong.to_numpy()].mean()],
    "% mixed opinion": [100 * flags.loc[err["type"] == "correct", MIXED].mean(), 100 * flags.loc[wrong, MIXED].mean()],
    "% with negation": [100 * flags.loc[err["type"] == "correct", "negation present"].mean(),
                        100 * flags.loc[wrong, "negation present"].mean()],
}, index=["correct", "wrong"])
display(cmp.round(1))

**Why the model fails on some reviews.** A Bag-of-Words model adds up word weights and does not understand *who* or *what* a word refers to. In a sarcastic review the positive words win the sum; in a mixed review both sides are present and the stronger words decide; a very short review gives the sum almost nothing to work with; and when the same text appears with both thumbs (D.3), no text-only model can be right every time. Fixing the first two needs a model that reads word order and context (see the future improvements below).

## D.7 Conclusions and answer to RQ1

In [ ]:
joblib.dump({"vectorizer": vectorizer, "model": final_model, "name": FINAL_NAME, "threshold": thr},
            C.MODELS_DIR / "D_final_text_model.joblib")

gain_lo, gain_hi = ci.loc["f1_macro_gain_vs_baseline", "2.5%"], ci.loc["f1_macro_gain_vs_baseline", "97.5%"]
ap_gain_lo = ci.loc["pr_auc_gain_vs_no_skill", "2.5%"]
rq1_yes = gain_lo > 0 and ap_gain_lo > 0
rq1 = ("YES — the 95% bootstrap interval of the gain is entirely above zero for both F1-macro and PR-AUC"
       if rq1_yes else "NOT SHOWN — the 95% bootstrap interval of the gain includes zero")
top_pattern = pattern_table["risk ratio"].idxmax()
hardest_len = by_len["error_rate"].idxmax()
bow_best = representation_table.loc[[n for n in representation_table.index if n.startswith("BoW")], "f1_macro"].astype(float).max()
tfidf_best = representation_table.loc[[n for n in representation_table.index if n.startswith("TF-IDF")], "f1_macro"].astype(float).max()
summary = [
    f"1. Preprocessing: cleaning from notebook 01; tokenisation and stop words decided by validation -> {BEST_REP}.",
    f"2. Representation: best BoW variant F1-macro {bow_best:.3f} vs best TF-IDF {tfidf_best:.3f}; "
    + "; ".join(f"{label}: {r['verdict (LR)']} ({r['F1 A − B (LR)']:+.3f})" for label, r in rep_findings.iterrows()) + ".",
    f"3. Cosine similarity: centroid similarity {centroid_cos:.3f}; {(dup['max cosine'] > 0.999).mean():.1%} of validation "
    f"reviews have an identical text in train; {100 * noise_rate:.1f}% of reviews with a frequent identical text carry the "
    "minority label (label noise).",
    "4. Validation F1-macro with tuned threshold: " + ", ".join(f"{n} {f['val']['f1_macro_tuned']:.3f}" for n, f in FITTED.items())
    + f"; majority baseline {baseline_val['f1_macro']:.3f}. Final model by the declared rule: {FINAL_NAME} {FITTED[FINAL_NAME]['params']}.",
    f"5. Overfitting: train–validation gap {FITTED[FINAL_NAME]['train']['f1_macro'] - FITTED[FINAL_NAME]['val']['f1_macro']:+.3f} "
    f"({gap_label(FITTED[FINAL_NAME]['train']['f1_macro'] - FITTED[FINAL_NAME]['val']['f1_macro'])}); learning curve 50%->100%: "
    f"{last_gain:+.3f} validation F1-macro.",
    (f"6. Tabular features on top of text (validation): {tab_result.loc['text + tabular', 'f1_macro_tuned'] - tab_result.loc['text only', 'f1_macro_tuned']:+.3f} F1-macro."
     if tab_result is not None else "6. Tabular features on top of text: see notebook 03 (check skipped here)."),
    f"7. Final text model {FINAL_NAME} on test: F1-macro {test_metrics['f1_macro']:.3f} "
    f"(95% CI {ci.loc['f1_macro', '2.5%']:.3f}–{ci.loc['f1_macro', '97.5%']:.3f}), recall(neg) {test_metrics['recall_neg']:.3f}, "
    f"precision(neg) {test_metrics['precision_neg']:.3f}, PR-AUC {test_metrics['pr_auc_neg']:.3f} vs no-skill {(y_test == 0).mean():.3f}.",
    f"8. RQ1: {rq1}. Gain over the majority baseline: {test_metrics['f1_macro'] - baseline_test['f1_macro']:+.3f} F1-macro "
    f"(95% CI {gain_lo:+.3f} to {gain_hi:+.3f}).",
    f"9. Errors: {(err['type'] != 'correct').mean():.1%} of test reviews; hardest length group '{hardest_len}' words; "
    f"strongest risk factor: {top_pattern} (risk ratio {pattern_table.loc[top_pattern, 'risk ratio']:.2f}).",
]
print("\n".join(summary))
(C.TABLES_DIR / "D_summary.txt").write_text("\n".join(summary), encoding="utf-8")

### Limitations and future improvements (for section 8 of the report)
**Limitations**
* **Bag-of-Words ignores word order and context** → sarcasm and mixed reviews remain an error source (see the risk ratios in D.6.3).
* **Label noise** (identical texts with opposite thumbs, D.3) caps the achievable accuracy of any text-only model.
* **English only** — the other 56 % of the reviews (other languages) are not covered.
* **Representation chosen with one classifier** (logistic regression); the Naive Bayes column in D.2 shows how much this matters.
* **Threshold chosen on validation** — the honest cross-fitted estimate is reported next to the optimistic one.
* **2 % sample** of the English reviews.

**Future improvements**
1. **Transformer model** (e.g. DistilBERT) fine-tuned on a subsample, compared with TF-IDF + linear model — reads word order and context, targets sarcasm and *"…, but …"* reviews.
2. **More data / other languages** — the learning curve (D.4.3) shows whether more reviews help; out-of-core training (`SGDClassifier`) on the full 9.6M English reviews; multilingual models for the other languages.
3. **Character n-grams** (`analyzer="char_wb"`) for typos and slang (*"gud"*, *"trash!!!"*).
4. **Label-noise handling** — flag reviews whose text strongly contradicts the label (D.6.4) and check a sample manually; train with those down-weighted.
5. **Stemming / lemmatisation** tested as one more representation in D.2.